In [3]:
import duckdb
import pandas as pd

con = duckdb.connect("../olist_db.duckdb")  # ../ because notebook is inside notebooks/

In [4]:
con.execute("""
    SELECT 
        MIN(order_purchase_timestamp) AS earliest_order,
        MAX(order_purchase_timestamp) AS latest_order
    FROM orders
""").fetchdf()

,earliest_order,latest_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [5]:
con.execute("""
    SELECT order_status, COUNT(*) AS count
    FROM orders
    GROUP BY order_status
    ORDER BY count DESC
""").fetchdf()

,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [6]:
con.execute("""
    SELECT 
        COUNT(*) AS total_orders,
        SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS missing_delivered_date,
        SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS missing_approved_date,
        SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS missing_estimated_date
    FROM orders
""").fetchdf()

,total_orders,missing_delivered_date,missing_approved_date,missing_estimated_date
0,99441,2965.0,160.0,0.0


In [7]:
con.execute("""
    SELECT 
        COUNT(*) AS total_reviews,
        SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS missing_score
    FROM order_reviews
""").fetchdf()

,total_reviews,missing_score
0,99224,0.0


In [8]:
con.execute("""
    SELECT 
        COUNT(*) AS delivered_status_orders,
        SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS still_missing_date
    FROM orders
    WHERE order_status = 'delivered'
""").fetchdf()

,delivered_status_orders,still_missing_date
0,96478,8.0


In [9]:
con.execute("""
    CREATE OR REPLACE TABLE orders_clean AS
    SELECT 
        *,
        DATE_DIFF('day', 
            order_purchase_timestamp::TIMESTAMP, 
            order_delivered_customer_date::TIMESTAMP
        ) AS delivery_days,
        DATE_DIFF('day', 
            order_purchase_timestamp::TIMESTAMP, 
            order_estimated_delivery_date::TIMESTAMP
        ) AS estimated_delivery_days,
        CASE 
            WHEN order_delivered_customer_date::TIMESTAMP > order_estimated_delivery_date::TIMESTAMP THEN 1 
            ELSE 0 
        END AS is_late
    FROM orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
""")

con.execute("SELECT COUNT(*) FROM orders_clean").fetchdf()

,count_star()
0,96470


In [10]:
con.execute("""
    SELECT 
        AVG(delivery_days) AS avg_delivery_days,
        AVG(estimated_delivery_days) AS avg_estimated_days,
        SUM(is_late) AS total_late_orders,
        ROUND(100.0 * SUM(is_late) / COUNT(*), 2) AS pct_late
    FROM orders_clean
""").fetchdf()

,avg_delivery_days,avg_estimated_days,total_late_orders,pct_late
0,12.496849,24.372738,7826.0,8.11


In [11]:
con.execute("""
    SELECT 
        c.customer_state,
        COUNT(*) AS total_orders,
        ROUND(AVG(oc.delivery_days), 2) AS avg_delivery_days,
        SUM(oc.is_late) AS late_orders,
        ROUND(100.0 * SUM(oc.is_late) / COUNT(*), 2) AS pct_late
    FROM orders_clean oc
    JOIN customers c ON oc.customer_id = c.customer_id
    GROUP BY c.customer_state
    ORDER BY pct_late DESC
""").fetchdf()

,customer_state,total_orders,avg_delivery_days,late_orders,pct_late
0,AL,397,24.50,95.0,23.93
1,MA,717,21.51,141.0,19.67
2,PI,476,19.39,76.0,15.97
3,CE,1279,21.20,196.0,15.32
4,SE,335,21.46,51.0,15.22
5,BA,3256,19.28,457.0,14.04
6,RJ,12350,15.24,1664.0,13.47
7,TO,274,17.60,35.0,12.77
8,PA,946,23.73,117.0,12.37
9,ES,1995,15.72,244.0,12.23


In [12]:
con.execute("""
    SELECT 
        oi.seller_id,
        COUNT(DISTINCT oc.order_id) AS total_orders,
        ROUND(AVG(oc.delivery_days), 2) AS avg_delivery_days,
        SUM(oc.is_late) AS late_orders,
        ROUND(100.0 * SUM(oc.is_late) / COUNT(DISTINCT oc.order_id), 2) AS pct_late,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM orders_clean oc
    JOIN order_items oi ON oc.order_id = oi.order_id
    LEFT JOIN order_reviews r ON oc.order_id = r.order_id
    GROUP BY oi.seller_id
    HAVING COUNT(DISTINCT oc.order_id) >= 5
    ORDER BY pct_late DESC
""").fetchdf()

,seller_id,total_orders,avg_delivery_days,late_orders,pct_late,avg_review_score
0,2709af9587499e95e803a6498a5a56e9,25,12.74,23.0,92.00,2.60
1,05e107217c7266362fd44b75b2cd4cc4,5,6.88,4.0,80.00,4.13
2,70125af26c2d6d4ef401a1d02ae7701f,5,10.25,4.0,80.00,4.63
3,973f21788dfab357250f69a8dcb7ddee,9,21.11,7.0,77.78,2.47
4,38e6dada03429a47197d5d584d793b41,7,12.67,5.0,71.43,3.33
...,...,...,...,...,...,...
1761,1b1ae47a313a825a7ccceb8e2e30fa9d,10,10.70,0.0,0.00,4.22
1762,a414555ce331b8c8aea4a9cb8395194d,7,7.63,0.0,0.00,3.57
1763,e256a62e0ac58c0edfba09966142d561,12,8.21,0.0,0.00,4.50
1764,de66a66f2dd06bb9ec37aa96987466a3,7,14.88,0.0,0.00,4.25


In [13]:
con.execute("""
    CREATE OR REPLACE TABLE orders_master AS
    SELECT 
        oc.*,
        oi.product_id,
        oi.seller_id,
        oi.price,
        oi.freight_value,
        (oi.price + oi.freight_value) AS item_total
    FROM orders_clean oc
    JOIN order_items oi ON oc.order_id = oi.order_id
""")

con.execute("SELECT COUNT(*) FROM orders_master").fetchdf()

,count_star()
0,110189


In [14]:
con.execute("""
    CREATE OR REPLACE TABLE order_revenue AS
    SELECT 
        order_id,
        SUM(item_total) AS order_revenue,
        COUNT(*) AS num_items
    FROM orders_master
    GROUP BY order_id
""")

con.execute("SELECT COUNT(*) FROM order_revenue").fetchdf()

,count_star()
0,96470


### Phase 3

In [15]:
con.execute("""
    SELECT 
        oi.seller_id,
        COUNT(DISTINCT oc.order_id) AS total_orders,
        ROUND(AVG(oc.delivery_days), 2) AS avg_delivery_days,
        SUM(oc.is_late) AS late_orders,
        ROUND(100.0 * SUM(oc.is_late) / COUNT(DISTINCT oc.order_id), 2) AS pct_late,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM orders_clean oc
    JOIN order_items oi ON oc.order_id = oi.order_id
    LEFT JOIN order_reviews r ON oc.order_id = r.order_id
    GROUP BY oi.seller_id
    HAVING COUNT(DISTINCT oc.order_id) >= 5
    ORDER BY pct_late DESC
""").fetchdf()

,seller_id,total_orders,avg_delivery_days,late_orders,pct_late,avg_review_score
0,2709af9587499e95e803a6498a5a56e9,25,12.74,23.0,92.00,2.60
1,05e107217c7266362fd44b75b2cd4cc4,5,6.88,4.0,80.00,4.13
2,70125af26c2d6d4ef401a1d02ae7701f,5,10.25,4.0,80.00,4.63
3,973f21788dfab357250f69a8dcb7ddee,9,21.11,7.0,77.78,2.47
4,c42fd8e4d47dfb18ce5222f2dd7752f9,7,34.36,5.0,71.43,4.00
...,...,...,...,...,...,...
1761,6b89abe95848c850399130d149a39b63,8,11.38,0.0,0.00,4.38
1762,02c988090b766852e088c69d7fb3b551,12,8.56,0.0,0.00,4.69
1763,41e0fa5761c886a630994a55c12087e7,5,10.20,0.0,0.00,5.00
1764,ef9952469137ff190bacafe117f51537,5,10.20,0.0,0.00,4.80


In [16]:
con.execute("""
    SELECT 
        DATE_TRUNC('month', om.order_purchase_timestamp::TIMESTAMP) AS month,
        t.product_category_name_english AS category,
        COUNT(*) AS order_count,
        SUM(om.item_total) AS revenue
    FROM orders_master om
    JOIN products p ON om.product_id = p.product_id
    JOIN product_category_translation t ON p.product_category_name = t.product_category_name
    GROUP BY month, category
    ORDER BY month, order_count DESC
""").fetchdf()

,month,category,order_count,revenue
0,2016-09-01,health_beauty,3,143.46
1,2016-10-01,furniture_decor,65,6899.35
2,2016-10-01,health_beauty,40,4186.29
3,2016-10-01,perfumery,29,5315.20
4,2016-10-01,toys,24,4591.53
...,...,...,...,...
1238,2018-08-01,fashio_female_clothing,3,169.13
1239,2018-08-01,books_imported,1,804.96
1240,2018-08-01,dvds_blu_ray,1,16.29
1241,2018-08-01,fashion_sport,1,83.05


In [17]:
con.execute("""
    SELECT 
        oc.is_late,
        COUNT(*) AS num_orders,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM orders_clean oc
    JOIN order_reviews r ON oc.order_id = r.order_id
    GROUP BY oc.is_late
""").fetchdf()

,is_late,num_orders,avg_review_score
0,0,88653,4.29
1,1,7700,2.57


In [18]:
con.execute("""
    SELECT 
        CORR(days_late, r.review_score) AS correlation
    FROM (
        SELECT 
            oc.order_id,
            DATE_DIFF('day', 
                oc.order_estimated_delivery_date::TIMESTAMP, 
                oc.order_delivered_customer_date::TIMESTAMP
            ) AS days_late
        FROM orders_clean oc
        WHERE oc.is_late = 1
    ) t
    JOIN order_reviews r ON t.order_id = r.order_id
""").fetchdf()

,correlation
0,-0.235061


In [19]:
con.execute("""
    SELECT 
        DATE_DIFF('day', 
            oc.order_estimated_delivery_date::TIMESTAMP, 
            oc.order_delivered_customer_date::TIMESTAMP
        ) AS days_late,
        COUNT(*) AS num_orders,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM orders_clean oc
    JOIN order_reviews r ON oc.order_id = r.order_id
    WHERE oc.is_late = 1
    GROUP BY days_late
    ORDER BY days_late
""").fetchdf()

,days_late,num_orders,avg_review_score
0,0,1291,4.03
1,1,823,3.73
2,2,537,3.18
3,3,496,2.68
4,4,437,2.49
...,...,...,...
109,165,1,1.00
110,166,1,1.00
111,167,1,1.00
112,175,1,1.00


In [20]:
rfm_base = con.execute("""
    SELECT 
        c.customer_unique_id,
        oc.order_id,
        oc.order_purchase_timestamp::TIMESTAMP AS order_date,
        orv.order_revenue
    FROM orders_clean oc
    JOIN customers c ON oc.customer_id = c.customer_id
    JOIN order_revenue orv ON oc.order_id = orv.order_id
""").fetchdf()

rfm_base.head()

,customer_unique_id,order_id,order_date,order_revenue
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,38.71
1,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,141.46
2,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,179.12
3,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,28.62
4,80bb27c7c16e8f973207a5086ab329e2,a4591c265e18cb1dcee52889e2d8acc3,2017-07-09 21:57:05,175.26


In [21]:
print("Unique orders:", rfm_base['order_id'].nunique())
print("Unique customers:", rfm_base['customer_unique_id'].nunique())

Unique orders: 96470
Unique customers: 93350


In [22]:
import pandas as pd
from datetime import datetime

# Reference date - one day after the last order in the dataset
reference_date = pd.Timestamp('2018-12-31')

rfm = rfm_base.groupby('customer_unique_id').agg(
    last_order_date=('order_date', 'max'),
    frequency=('order_id', 'nunique'),
    monetary=('order_revenue', 'sum')
).reset_index()

rfm['recency'] = (reference_date - rfm['last_order_date']).dt.days

rfm.head()

,customer_unique_id,last_order_date,frequency,monetary,recency
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,1,141.90,234
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,1,27.19,237
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,1,86.22,660
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,1,43.62,444
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,1,196.89,411


In [23]:
# Recency: lower days = better = higher score, so we reverse the labels
rfm['R_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1])

# Frequency: many customers will tie at frequency=1, which can break qcut (duplicate bin edges)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# Monetary: higher spend = better = higher score
rfm['M_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5])

rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

rfm.head(10)

,customer_unique_id,last_order_date,frequency,monetary,recency,R_score,F_score,M_score,RFM_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,1,141.90,234,4,1,4,9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,1,27.19,237,4,1,1,6
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,1,86.22,660,1,1,2,4
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,1,43.62,444,2,1,1,4
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,1,196.89,411,2,1,4,7
5,0004bd2a26a76fe21f786e4fbd80607f,2018-04-05 19:33:16,1,166.98,269,4,1,4,9
6,00050ab1314c0e55a6ca13cf7181fecf,2018-04-20 12:57:23,1,35.38,254,4,1,1,6
7,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:41,1,419.18,305,3,1,5,9
8,0005e1862207bf6ccc02e4228effd9a0,2017-03-04 23:32:12,1,150.12,666,1,1,4,6
9,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:12,1,129.76,293,4,1,3,8


In [24]:
rfm['F_score_binary'] = rfm['frequency'].apply(lambda x: 1 if x == 1 else 5)

rfm['F_score_binary'].value_counts()

F_score_binary
1    90549
5     2801
Name: count, dtype: int64

In [25]:
def score_frequency(freq, repeat_customers_frequencies):
    if freq == 1:
        return 1
    else:
        # Rank only among repeat customers (freq >= 2)
        pass

# Simpler implementation: separate repeat customers, qcut only them
repeat_mask = rfm['frequency'] > 1

rfm['F_score_tiered'] = 1  # default: everyone starts at 1 (one-time buyers)

# For repeat customers, rank into up to 4 additional tiers (2-5)
repeat_freqs = rfm.loc[repeat_mask, 'frequency']
rfm.loc[repeat_mask, 'F_score_tiered'] = pd.qcut(
    repeat_freqs.rank(method='first'), q=4, labels=[2, 3, 4, 5]
).astype(int)

rfm['F_score_tiered'].value_counts().sort_index()

F_score_tiered
1    90549
2      701
3      700
4      700
5      700
Name: count, dtype: int64

In [26]:
rfm['F_score'] = rfm['F_score_tiered']  # replace the old F_score

rfm['RFM_Score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

rfm[['customer_unique_id', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_Score']].head(10)

,customer_unique_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,234,1,141.90,4,1,4,9
1,0000b849f77a49e4a4ce2b2a4ca5be3f,237,1,27.19,4,1,1,6
2,0000f46a3911fa3c0805444483337064,660,1,86.22,1,1,2,4
3,0000f6ccb0745a6a4b88665a16c9f078,444,1,43.62,2,1,1,4
4,0004aac84e0df4da2b147fca70cf8255,411,1,196.89,2,1,4,7
5,0004bd2a26a76fe21f786e4fbd80607f,269,1,166.98,4,1,4,9
6,00050ab1314c0e55a6ca13cf7181fecf,254,1,35.38,4,1,1,6
7,00053a61a98854899e70ed204dd4bafe,305,1,419.18,3,1,5,9
8,0005e1862207bf6ccc02e4228effd9a0,666,1,150.12,1,1,4,6
9,0005ef4cd20d2893f0d9fbd94d3c0d97,293,1,129.76,4,1,3,8


In [27]:
rfm['RFM_Score'].describe()

count    93350.000000
mean         7.082121
std          2.125457
min          3.000000
25%          6.000000
50%          7.000000
75%          9.000000
max         15.000000
Name: RFM_Score, dtype: float64

In [28]:
rfm['RFM_Score'].value_counts().sort_index()

RFM_Score
3      3877
4      7434
5     11473
6     14122
7     17928
8     14629
9     11097
10     7680
11     3979
12      462
13      338
14      221
15      110
Name: count, dtype: int64

In [29]:
def segment_customer(score):
    if score >= 13:
        return 'Champions'
    elif score >= 10:
        return 'Loyal Customers'
    elif score >= 6:
        return 'At Risk'
    else:
        return 'Lost'

rfm['segment'] = rfm['RFM_Score'].apply(segment_customer)

rfm['segment'].value_counts()

segment
At Risk            57776
Lost               22784
Loyal Customers    12121
Champions            669
Name: count, dtype: int64

In [30]:
segment_revenue = rfm.groupby('segment')['monetary'].agg(
    total_revenue='sum',
    customer_count='count',
    avg_revenue='mean'
).reset_index()

segment_revenue['pct_of_total_revenue'] = round(
    100 * segment_revenue['total_revenue'] / rfm['monetary'].sum(), 2
)

segment_revenue.sort_values('total_revenue', ascending=False)

,segment,total_revenue,customer_count,avg_revenue,pct_of_total_revenue
0,At Risk,9545002.49,57776,165.207049,61.91
3,Loyal Customers,4198876.38,12121,346.413364,27.23
2,Lost,1399733.34,22784,61.434925,9.08
1,Champions,274782.62,669,410.736353,1.78


In [31]:
# Average Order Value per customer
rfm['avg_order_value'] = rfm['monetary'] / rfm['frequency']

# Purchase Frequency = number of orders (you already have this)
# rfm['frequency'] already exists

# Average Customer Lifespan — for one-time buyers this is tricky since there's only one purchase date.
# Common approach: use time between first and last order as "lifespan" in years,
# but this requires first_order_date too. Let's pull that in.

rfm_dates = con.execute("""
    SELECT 
        c.customer_unique_id,
        MIN(oc.order_purchase_timestamp::TIMESTAMP) AS first_order_date,
        MAX(oc.order_purchase_timestamp::TIMESTAMP) AS last_order_date
    FROM orders_clean oc
    JOIN customers c ON oc.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
""").fetchdf()

rfm = rfm.merge(rfm_dates[['customer_unique_id', 'first_order_date']], on='customer_unique_id', how='left')

# Lifespan in days (will be 0 for one-time buyers)
rfm['lifespan_days'] = (rfm['last_order_date'] - rfm['first_order_date']).dt.days
rfm['lifespan_days'].describe()

count    93350.000000
mean         2.634258
std         24.956879
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        633.000000
Name: lifespan_days, dtype: float64

In [32]:
# Average lifespan among repeat customers only (excludes 0-lifespan one-time buyers)
repeat_customers = rfm[rfm['frequency'] > 1]
avg_lifespan_days = repeat_customers['lifespan_days'].mean()
avg_lifespan_years = avg_lifespan_days / 365

print(f"Avg lifespan among repeat customers: {avg_lifespan_days:.1f} days ({avg_lifespan_years:.2f} years)")

Avg lifespan among repeat customers: 87.8 days (0.24 years)


In [33]:
rfm['CLV'] = rfm['avg_order_value'] * rfm['frequency'] * avg_lifespan_years

rfm.groupby('segment')['CLV'].agg(
    avg_CLV='mean',
    median_CLV='median',
    total_CLV='sum'
).reset_index().sort_values('avg_CLV', ascending=False)

,segment,avg_CLV,median_CLV,total_CLV
1,Champions,98.793831,72.976371,6.609307e+04
3,Loyal Customers,83.322314,57.907255,1.009950e+06
0,At Risk,39.737017,28.295782,2.295846e+06
2,Lost,14.776855,13.395037,3.366759e+05


In [34]:
rfm['is_churned'] = (rfm['recency'] > 180).astype(int)

rfm['is_churned'].value_counts()

is_churned
1    81325
0    12025
Name: count, dtype: int64

In [35]:
customer_state_map = con.execute("""
    SELECT DISTINCT customer_unique_id, customer_state
    FROM customers
""").fetchdf()

rfm = rfm.merge(customer_state_map, on='customer_unique_id', how='left')

churn_by_state = rfm.groupby('customer_state')['is_churned'].agg(
    total_customers='count',
    churned_customers='sum'
).reset_index()

churn_by_state['churn_rate_pct'] = round(100 * churn_by_state['churned_customers'] / churn_by_state['total_customers'], 2)

churn_by_state.sort_values('churn_rate_pct', ascending=False)

,customer_state,total_customers,churned_customers,churn_rate_pct
0,AC,76,69,90.79
12,MT,856,776,90.65
9,MA,700,633,90.43
1,AL,388,350,90.21
20,RO,231,208,90.04
19,RN,464,416,89.66
16,PI,464,414,89.22
13,PA,922,822,89.15
5,CE,1258,1121,89.11
23,SC,3449,3069,88.98


In [36]:
churn_by_segment = rfm.groupby('segment')['is_churned'].agg(
    total_customers='count',
    churned_customers='sum'
).reset_index()

churn_by_segment['churn_rate_pct'] = round(100 * churn_by_segment['churned_customers'] / churn_by_segment['total_customers'], 2)

churn_by_segment.sort_values('churn_rate_pct', ascending=False)

,segment,total_customers,churned_customers,churn_rate_pct
2,Lost,22784,22784,100.00
0,At Risk,57783,50686,87.72
1,Champions,685,446,65.11
3,Loyal Customers,12138,7438,61.28


In [37]:
# Find the duplicated customers to see what's happening
dupes = rfm[rfm.duplicated('customer_unique_id', keep=False)]
dupes[['customer_unique_id', 'customer_state']].sort_values('customer_unique_id').head(20)

,customer_unique_id,customer_state
11589,1f90117a847636892e3c5bf569f2ac68,PR
11590,1f90117a847636892e3c5bf569f2ac68,SC
13223,2410195f6521688005612363835a2671,RS
13224,2410195f6521688005612363835a2671,SP
16170,2c45ab66a3dae52960147e76a35740ff,RS
16171,2c45ab66a3dae52960147e76a35740ff,MS
16220,2c6a91479a7dc00d8c9d650d8dee88ca,SC
16221,2c6a91479a7dc00d8c9d650d8dee88ca,PR
23584,408aee96c75632a92e5079eee61da399,SP
23583,408aee96c75632a92e5079eee61da399,RJ


In [38]:
# Rebuild the state map with one row per customer — keep the most recent order's state
customer_state_map = con.execute("""
    SELECT customer_unique_id, customer_state
    FROM (
        SELECT 
            c.customer_unique_id,
            c.customer_state,
            oc.order_purchase_timestamp::TIMESTAMP AS order_date,
            ROW_NUMBER() OVER (
                PARTITION BY c.customer_unique_id 
                ORDER BY oc.order_purchase_timestamp::TIMESTAMP DESC
            ) AS rn
        FROM orders_clean oc
        JOIN customers c ON oc.customer_id = c.customer_id
    )
    WHERE rn = 1
""").fetchdf()

print("Rows:", len(customer_state_map))
print("Unique customers:", customer_state_map['customer_unique_id'].nunique())

Rows: 93350
Unique customers: 93350


In [39]:
rfm = rfm.drop(columns=['customer_state'])
rfm = rfm.merge(customer_state_map, on='customer_unique_id', how='left')

print("Total rows in rfm:", len(rfm))

Total rows in rfm: 93390


In [40]:
churn_by_state = rfm.groupby('customer_state')['is_churned'].agg(
    total_customers='count',
    churned_customers='sum'
).reset_index()

churn_by_state['churn_rate_pct'] = round(100 * churn_by_state['churned_customers'] / churn_by_state['total_customers'], 2)

churn_by_state.sort_values('churn_rate_pct', ascending=False)

,customer_state,total_customers,churned_customers,churn_rate_pct
0,AC,76,69,90.79
12,MT,856,776,90.65
9,MA,702,635,90.46
1,AL,387,349,90.18
20,RO,232,209,90.09
19,RN,464,416,89.66
16,PI,464,414,89.22
13,PA,922,822,89.15
5,CE,1261,1124,89.14
23,SC,3450,3072,89.04


In [41]:
basket_data = con.execute("""
    SELECT 
        om.order_id,
        t.product_category_name_english AS category
    FROM orders_master om
    JOIN products p ON om.product_id = p.product_id
    JOIN product_category_translation t ON p.product_category_name = t.product_category_name
""").fetchdf()

basket_data.head()

,order_id,category
0,e481f51cbdc54678b7cc49136f2d6af7,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,stationery


In [42]:
from itertools import combinations
from collections import Counter

pair_counts = Counter()

for order_id, group in basket_data.groupby('order_id'):
    categories = sorted(set(group['category']))
    if len(categories) > 1:
        for pair in combinations(categories, 2):
            pair_counts[pair] += 1

top_pairs = pd.DataFrame(pair_counts.items(), columns=['category_pair', 'count']).sort_values('count', ascending=False)
top_pairs.head(10)

,category_pair,count
22,"(bed_bath_table, furniture_decor)",70
80,"(bed_bath_table, home_confort)",43
27,"(furniture_decor, housewares)",24
3,"(bed_bath_table, housewares)",20
9,"(baby, cool_stuff)",20
1,"(baby, toys)",19
44,"(baby, bed_bath_table)",17
42,"(furniture_decor, garden_tools)",16
74,"(health_beauty, sports_leisure)",14
21,"(furniture_decor, home_construction)",13


In [43]:
import os

os.makedirs('../exports', exist_ok=True)

# Orders with delivery metrics — for Supply Chain dashboard
orders_clean_df = con.execute("SELECT * FROM orders_clean").fetchdf()
orders_clean_df.to_csv('../exports/orders_clean.csv', index=False)

# Line-item level master table — for category/seller revenue analysis
orders_master_df = con.execute("SELECT * FROM orders_master").fetchdf()
orders_master_df.to_csv('../exports/orders_master.csv', index=False)

# Order-level revenue
order_revenue_df = con.execute("SELECT * FROM order_revenue").fetchdf()
order_revenue_df.to_csv('../exports/order_revenue.csv', index=False)

# Product category translation (for readable category names in Tableau)
categories_df = con.execute("SELECT * FROM product_category_translation").fetchdf()
categories_df.to_csv('../exports/product_categories.csv', index=False)

# Sellers, customers (for geo joins in Tableau)
sellers_df = con.execute("SELECT * FROM sellers").fetchdf()
sellers_df.to_csv('../exports/sellers.csv', index=False)

customers_df = con.execute("SELECT * FROM customers").fetchdf()
customers_df.to_csv('../exports/customers.csv', index=False)

# RFM/CLV/churn table — for Customer Insights dashboard
rfm.to_csv('../exports/rfm_segments.csv', index=False)

print("All exports complete.")

All exports complete.


In [44]:
products_df = con.execute("SELECT * FROM products").fetchdf()
products_df.to_csv('../exports/products.csv', index=False)
print("products.csv exported")

products.csv exported
